<h1 align="center"> Práctica 0: Nodos y Tópicos</h1>
<p style="text-align: center;"> NANCY MANZANO MARTÍNEZ </p>

## Objetivo
Los alumnos aprenderan las bases de la gestión de información e interacción relacionadas con el manejo de robots en entornos virtuales y en robots reales.

## Previo de la Práctica
### La estructura de paquetes en ROS 2.


En ROS un paquete es una unidad de software organizativa que incluye el código fuente, los archivos de compilación, la documentación, las pruebas, y otros recursos asociados.

Un único espacio de trabajo puede contener más de un paquete, del tipo de compilación que sea. Lo recomendable, y como se ha trabajado para esta práctica, es tener una carpeta SRC dentro del espacio de trabajo, en donde se encuentren los paquetes utilizados.

La creación de paquetes en ROS 2 utiliza ament como sistema de compilación y colcon como herramienta de compilación. Los contenidos en la estructura de los paquetes de ROS2 dependen del tipo de construcción utilizado: Cmake o python. En este caso se presenta la estructura referente a python ya que la práctica ha sido realizada de esta manera.

Estructura mínima necesaria para paquetes de python:
```bash
- package.xml 
- resource/<package_name>
- setup.cfg 
- setup.py
- <package_name> 
     __init__.py
- Test
```
En donde, package.xml contiene la información más importante del paquete, resource registra el paquete en el sistema y guarda su nombre, setup.cfg es necesario para que ROS2 pueda encontrar los ejecutables del paquete, setup.py contiene instrucciones de instalación, y package_name contiene al código principal. Por último, Test es una capeta destinada a pruebas.

Para el caso de esta práctica, la estructura del paquete es la siguiente:

```bash
p0_py/
├── package.xml
├── resource/p0_py
├── setup.cfg
├── setup.py
├── p0_py/
│   ├── __init__.py
│   ├── primer_nodo.py
│   └── subscriber_nodo.py
└── test/
```
      


### Definición de Nodos y Tópicos.

Con base en la documentación oficial de ROS, en específico la de ROS2, se obtienen las siguientes definiciones:

Un NODO es un elemento fundamental de ROS que cumple una _única_ función modular en un sistema robótico. Los nodos forman parte de una red de ROS llamada ROS graph, compuesta de elementos que procesan datos al mismo tiempo. Cada nodo es capaz de mandar y recibir datos de otros nodos del sistema, a través de topics, serivicios, acciones o parámetros. De esta forma, un sistema robótico está compuesto de la interacción de muchos nodos trabajando al mismo tiempo.

Por otro lado, cada nodo tiene un nombre único dentro del sistema que le permite ser identificado, y además, tiene un tipo de nodo que corresponde la archivo ejecutable que permite correrlo. 

Los nodos _publican_ y se _suscriben_ a Topics, mientras un nodo publica información, otro nodo debe subscribirse al topic para recibirla. Los nodos _ofrecen_ y _consumen_ servicios, es decir, un nodo envía una solicitud a otro nodo y espera una respuesta inmediata. Por último, los nodos _ejecutan_ acciones, que permiten enviar un objetivo a otro nodo y recibir retroalimentación junto con un resultado final.

Lo anterior se puede visualizar de manera gráfica en la figura 1.

Los topics en el grafo de ROS son los canales de comunicación que permiten que los nodos intercambien información, es decir, funcionan como un bus de datos. Un nodo puede publicar información a más de un topic, así como subscribirse simultáneamente a más de uno. Estos están diseñados para la comunicación contínua y unidireccional de datos, cada topic está fuertemente tipado por el tipo de mensaje utilizado para publicar en él, esto es para garantizar que los nodos subscriptores sólo puedan recibir mensajes del mismo tipo.




<p align="center">
<img src="Imagenes_p0/NodesTopics.png" width="700">
</p>

<p style="text-align: center;"> <b>Figura 1.</b> Representación de nodos, tópicos y servicios en ROS 2.
Fuente: [2]</p>

## Desarrollo de la práctica

Para el desarrollo de esta práctica se implementaron dos nodos en ROS2, escritos en python. El primer nodo es publicador, y publica valores entre cero y uno, correspondientes a una señal senoidal que simula velocidad en RPM de un motor. El segundo nodo es subscriptor-publicador, este recibe los valores de velocidad RPM y los publica ahora en unidades de rad/s.

Para generar el primer nodo publicador, se define una clase MyNode, inicializando un nodo llamado my_node. Dentro de este nodo se crea un publicador que envía un mensaje de tipo float64 a través del topico_1, así como un temporizador que ejecuta una función callback cada segundo. En cada ejecución, el nodo incrementa un contador, que es usado para calcular el valor de la función senoidal ajustada para oscilar entre 0 y 1, lo cual representa las RPM normalizadas del motor. Recordando que la función seno original oscila entre -1 y 1, para este caso se tuvo que multiplicar por 0.5 para reducirla a la mitad, y sumarle 0.5 para subir la función sobre el eje de las ordenadas; de tal manera que la función resultante es "0.5*sen(x)+0.5" y oscila entre 0 y 1. (En el código se ha reducido el periodo de la función, y se le han pedido 20 impresiones por periodo.)

Este valor es asignado al mensaje y publicado en el tópico, permitiendo que el nodo subscriptor lo reciba. Finalmente el programa inicializa el sistema en ROS2, mantiene el nodo en ejecución continua mediante rclpy.spin() y lo cierra cuando termina.

Por otro lado, se genera el nodo subscriptor-publicador, quien se subscribe aal tópico "topic_1", donde recibe mensajes de tipo float64 del nodo publicador. Cada vez que un mensaje llega, se ejecuta una función de callback que toma el valor recibido y realiza una conversión de unidades de RPM a rad/s, utilizando la relación RPM*(2pi)/60. El resultado de esta conversión se almacena en la variable self.counter y se asigna a un nuevo mensaje del mismo tipo. Posteriormente, el nuevo mensaje con las unidades convertidas es publicado en otro tópico llamado counter_topic. 


## Código del Nodo Publicador

```python
#!urs/bin/env python3
#Este nodo publica velocidad RPM de un motor, en términos de una función senoidal, con valores que varían de 0 a 1. 
#Se lo manda a un subscriptor por medio del tópico 1

import math
import rclpy
from rclpy.node import  Node
from std_msgs.msg import Float64

class MyNode(Node):
    def __init__(self):
        super().__init__("my_node")
        self.contador=0
        self.publicador_ = self.create_publisher(msg_type = Float64,topic ="topic_1",qos_profile= 10)
        self.timer_ = self.create_timer(timer_period_sec= 1.0, callback= self.cbck)
        self.get_logger().info("Activate node")

    def funcion_senoidal(self,contador): 
        #0.5*sen((2*pi/20)x)+0.5 -> 20 Pasos por periodo o 10 entre valle y cresta
        #simplificación: 0.5*sen((pi/10)x)+0.5
        seno=0.5*math.sin((math.pi/10)*contador)+0.5
        return seno
        
    def cbck(self):
        msg = Float64()
        self.contador=self.contador+1
        msg.data = self.funcion_senoidal(self.contador)
        self.publicador_.publish(msg)

def main(args= None):
    rclpy.init(args=args)
    node = MyNode()
    rclpy.spin(node)
    rclpy.shutdown()

if __name__ == '__main__':
    main()
```

## Código del Nodo Subscriptor-Publicador

```python
#!urs/bin/env python3
# Este nodo recibe valores entre 0 y 1 en valores tipo Float64
# Ademas, los transforma de RPM a rad/s y los publica a través del tópico counter_topic

import math
import rclpy
from rclpy.node import  Node
from std_msgs.msg import Float64

class NodeCounter(Node):
    def __init__(self):
        super().__init__("subscribe_node")

        self.counter_ = 0
        self.create_subscription(msg_type= Float64, callback= self.sub_cbck,topic="topic_1", qos_profile= 10)
        self.publisher_counter_ = self.create_publisher(msg_type= Float64,topic='counter_topic',qos_profile= 10)
        self.get_logger().info("Nodo subscriptor activo")
        
    def sub_cbck(self, msg):
        #Conversión de RPM a rad/s = RPM*2pi/60 = RPM*pi/30
        self.counter_ = (msg.data*math.pi)/30 #Conversion unidades
        new_msg = Float64()
        new_msg.data = self.counter_
        self.publisher_counter_.publish(new_msg) 

def main(args= None):
    rclpy.init(args=args)
    node = NodeCounter()
    rclpy.spin(node)
    rclpy.shutdown()

if __name__ == '__main__':
    main()
```

## Resultados

La figura 3 muestra las salidas en terminal cuando se activan ambos nodos y se visualizan sus respectivos tópicos. La terminal del lado izquierdo imprime las publicaciones del nodo publicador, con valores en RPM. La terminal del lado derecho imprime las publicaciones del nodo subscriptor-publicador, con valores en rad/s. 

<p align="center">
  <img src="Imagenes_p0/TerminalesTopicos.png" width="700">
</p>

<p style="text-align: center;"> <b>Figura 2.</b> Valores publicados por ambos nodos.</p>

La figura 2 muestra la gráfica que se forma de la lista de valores publicados por el nodo publicador. En esta figura sólo se muestra lo equivalente a un periodo de la señal, sin embargo, la publicación de valores es periódica. 

<p align="center">
  <img src="Imagenes_p0/ValoreTopic_1.png" width="700">
</p>

<p style="text-align: center;"> <b>Figura 3.</b> Valores publicados por el nodo publicador.</p>

La figura 3 muestra los valores publicados por el nodo subscriptor-publicador, de forma gráfica. Nótese que al igual que los publicados por el primer nodo, estos forman una onda senoidal, y también se publican 20 puntos por periodo de la señal.

<p align="center">
  <img src="Imagenes_p0/ValoresCounter_Topic_Corregido.png" width="700">
</p>

<p style="text-align: center;"> <b>Figura 4.</b> Valores publicados por el nodo subscriptor-publicador.</p>

## Video funcionamiento de la práctica

El siguiente video muestra la salida en terminales de los nodos publicador y subscriptor. El nodo publicador es llamado primer_nodo, y el nodo subscriptor-publicador es llamado subscriptor_nodo. Los topics en que estos publican son topic_1 y counter_topic, respectivamente. 

En pantalla se muestran cuatro terminales, las dos superiores corresponden a las termianles en que se activan ambos nodos, y las dos terminales inferiores corresponden a las publicaciones de estos nodos en sus respectivos topics. El topic_1 se muestra en la esquina inferior izquierda y publica datos variables entre cero y uno, correspondientes a velocidades en valores RPM; el encounter_topic se muestra en la esquina inferior derecha y publica los datos que recibe del nodo 1 pero convertidos en velocidades con unidades en rad/s. 

Nótese que los valores entre cero y uno forman una función senoidal, que ha sido dividida en veinte puntos por periodo, es decir, entre valle y cresta de la función el nodo publicador manda 10 valores al subscriptor. Esto se ha programado así para que la cantidad de datos que se manda por cada periodo de la señal no sea excesiva, pero que tampoco perjudique la forma de onda senoidal. 

Link de acceso al video: https://drive.google.com/file/d/1jwEdUNWgF-u7hhD6Sh_u_eEAuDScQvnp/view?usp=sharing

## Conclusiones

Se comprendió con éxito la arquitectura de comunicación en ROS2, específicamente en lo que respecta a la implementación de nodos y tópicos. De esta manera se simuló el flujo de imformación característico de los sistemas robóticos. En este caso, se logró implementar con éxito un nodo publicador que publicara valores de una función senoidal representativos de una velocidad en RPM, de igual forma se logró implementar con éxito al nodo subscriptor-publicador que recibiera estos valores y los convirtiera a unidades rad/s, republicándolos en un tópico diferente.

La visualización de los resultados en las terminales, correspondientes a cada tópico, confirman el correcto flujo de información entre nodos, así como la efectividad de la conversión de unidades, manteniendo la forma de onda senoidal original. Además, graficar los valores impresos en ambos tópicos permitió visualizar el correcto comportamiento de la señal, y que los 20 valores por periodo son suficientes para representar la forma de onda senoidal. 

La practica cero resulta fundamental para sentar las bases del desarrollo de sistemas robóticos en ROS2, se destaca de ésta la importancia de la correcta comunicación entre nodos, para poder controlar y monitorear robots en entornos ya sea virtuales o reales.

## Referencias

[1] Anis Koubaa, Ed., Robot Operating System (ROS): The Complete Reference (Volume 1). Cham, Switzerland: Springer, 2016.

[2] ROS 2 Documentation, “Understanding ROS 2 Nodes.” [En línea]. Disponible en: https://docs.ros.org/en/kilted/Tutorials/Beginner-CLI-Tools/Understanding-ROS2-Nodes/Understanding-ROS2-Nodes.html

[3] ROS 2 Documentation, “Understanding ROS 2 Topics.” [En línea]. Disponible en: https://docs.ros.org/en/kilted/Tutorials/Beginner-CLI-Tools/Understanding-ROS2-Topics/Understanding-ROS2-Topics.html

[4] ROS 2 Documentation, “Creating Your First ROS 2 Package.” [En línea]. Disponible en: https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Creating-Your-First-ROS2-Package.html

[5] ROS Wiki, “Nodes.” [En línea]. Disponible en: https://wiki.ros.org/Nodes

[6] ROS Wiki, “Topics.” [En línea]. Disponible en: https://wiki.ros.org/Topics